# 🏛️ India Legal & Policy Data Ingestion

| Section | What it does |
|---|---|
| 0 | Install dependencies |
| 1 | Config & helpers |
| 2 | Unity Catalog Volume setup |
| 3a | data.gov.in — discover datasets |
| 3b | data.gov.in — ingest all records |
| 4a | DAKSH — portal API |
| 4b | DAKSH — HTML scrape |
| 5 | PRS MP activity |
| 6a | PRS Bills — open session |
| 6b | PRS Bills — AJAX fetch |
| 6c | PRS Bills — save / fallback |
| 7a | PRS Acts — open session |
| 7b | PRS Acts — AJAX fetch |
| 7c | PRS Acts — save |
| 8 | Legal PDFs — BNS / Constitution |
| 9a | BNS CSV — GitHub mirror |
| 9b | BNS CSV — PDF fallback |
| 9c | BNS — save sections |
| 9d | BNS — IPC mapping |
| 10 | Gov Schemes — MyScheme |
| 11 | Build RAG corpus |
| 12 | Manual upload helper |
| 13 | Sample SQL queries |

## 0. Install dependencies
Run once per cluster/session restart.

In [0]:
%pip install -q requests pandas beautifulsoup4 lxml openpyxl pymupdf
print('✅ Packages installed')

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Packages installed


## 1. Config & shared helpers
**Edit `DATAGOV_API_KEY` before running anything else.**

In [0]:
# Secrets: create scope `nyaya-dhwani` in Databricks (User Settings → Secrets) or set env vars locally.
import os

def _read_secret(env_name: str, scope_key: str = None):
    """Databricks secret scope `nyaya-dhwani`, else upper-case env var."""
    try:
        return dbutils.secrets.get(scope="nyaya-dhwani", key=scope_key or env_name.lower())  # noqa: F821
    except Exception:
        return os.environ.get(env_name, "")

DATAGOV_API_KEY = _read_secret("DATAGOV_API_KEY", "datagov_api_key")
CATALOG  = 'main'
SCHEMA   = 'india_legal'
VOLUME   = 'legal_files'

import re, time, requests
import pandas as pd
from io       import StringIO, BytesIO
from datetime import datetime
from bs4      import BeautifulSoup

HEADERS  = {'User-Agent': 'Mozilla/5.0 (research/educational use)'}
VOL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'
PDF_DIR  = f'{VOL_PATH}/pdfs'

def clean_cols(df):
    df.columns = [
        re.sub(r'[\s,;{}()\n\t=]+', '_', str(c)).strip('_')
        for c in df.columns
    ]
    return df

def save_table(df, table):
    df  = clean_cols(df)
    sdf = spark.createDataFrame(df.astype(str))
    sdf.write.format('delta').mode('overwrite') \
       .option('overwriteSchema', 'true') \
       .saveAsTable(f'{CATALOG}.{SCHEMA}.{table}')
    print(f'  💾 {CATALOG}.{SCHEMA}.{table}  ({df.shape[0]} rows × {df.shape[1]} cols)')

if not DATAGOV_API_KEY:
    print('⚠️  DATAGOV_API_KEY is empty — set env DATAGOV_API_KEY or secret nyaya-dhwani/datagov_api_key')
else:
    print('✅ Config loaded (API key present)')
print(f'   Volume : {VOL_PATH}')


✅ Config loaded
   Volume : /Volumes/main/india_legal/legal_files


## 2. Unity Catalog setup
Creates the schema and Volume. Safe to re-run.

In [0]:
spark.sql(f'CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}')
spark.sql(f'CREATE VOLUME  IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}')
os.makedirs(PDF_DIR, exist_ok=True)
print(f'✅ Schema : {CATALOG}.{SCHEMA}')
print(f'✅ Volume : {VOL_PATH}')
print(f'✅ PDF dir: {PDF_DIR}')

✅ Schema : main.india_legal
✅ Volume : /Volumes/main/india_legal/legal_files
✅ PDF dir: /Volumes/main/india_legal/legal_files/pdfs


## 3a. data.gov.in — Discover datasets
Searches by keyword, probes each result for record count.
Saves to `datagov_catalog` — use resource IDs in 3b.

In [0]:
# ── Cell 3a: Legal & Policy Data — API + Direct GitHub CSV Downloads ─────────
#
# HONEST STATUS of data.gov.in:
#   • The /lists search API silently ignores all keywords → always empty
#   • The /apis portal page now shows "0 APIs" — discovery UI is broken
#   • Most NCRB/MoJ datasets have NO API; they are download-only
#   • Only 2 confirmed working API resources for legal/policy data:
#       1. ncrb_ipc_crime_heads    (36 rows — summary table)
#       2. pending_court_cases     (3,329 rows — police station level)
#
# STRATEGY:
#   Part A — Fetch the 2 confirmed data.gov.in APIs
#   Part B — Download rich NCRB CSV datasets directly from GitHub mirrors
#            (same data, sourced from official NCRB publications, open license)
#   Part C — Print manual download guide for datasets requiring login/Kaggle
# ─────────────────────────────────────────────────────────────────────────────

# ═══════════════════════════════════════════════════════════════════════════
# PART A — data.gov.in confirmed API resources
# ═══════════════════════════════════════════════════════════════════════════

CONFIRMED_APIS = {
    'b031d51c-e7db-4ef6-a47a-08ee97fd9f62': 'ncrb_ipc_crime_heads',
    '3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69': 'pending_court_cases',
}

def fetch_all_datagov(resource_id, label, max_records=50000):
    base = f'https://api.data.gov.in/resource/{resource_id}'
    recs, offset = [], 0
    while offset < max_records:
        try:
            r = requests.get(base, params={
                'api-key': DATAGOV_API_KEY, 'format': 'json',
                'limit': 100, 'offset': offset
            }, timeout=30, headers=HEADERS)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f'  ⚠️  offset {offset}: {e}'); break
        batch = data.get('records', [])
        if not batch: break
        recs.extend(batch)
        total = int(data.get('total', 0))
        offset += 100
        print(f'  {label}: {len(recs):,}/{min(total,max_records):,}', end='\r')
        time.sleep(0.25)
        if offset >= total: break
    print(f'\n  ✅ {label}: {len(recs):,} records')
    return pd.DataFrame(recs) if recs else pd.DataFrame()

print('═' * 60)
print('PART A — data.gov.in confirmed APIs')
print('═' * 60)
for rid, tbl in CONFIRMED_APIS.items():
    print(f'\n📥 {tbl}')
    df = fetch_all_datagov(rid, tbl)
    if not df.empty:
        display(df.head(3))
        save_table(df, tbl)


# ═══════════════════════════════════════════════════════════════════════════
# PART B — NCRB datasets via GitHub mirrors (same data, open license)
#
# All files are the official NCRB "Crime in India" publication data,
# mirrored on GitHub by researchers under the Govt Open Data License.
# Source organisation: National Crime Records Bureau, MHA, Govt of India.
# ═══════════════════════════════════════════════════════════════════════════

GITHUB_DATASETS = {

    # ── IPC Crimes — District-wise 2001-2012 ─────────────────────────────
    # 30 crime categories × all districts × 12 years ≈ 75,000+ rows
    # Cols: STATE/UT, DISTRICT, YEAR, MURDER, RAPE, KIDNAPPING, DACOITY,
    #       ROBBERY, BURGLARY, THEFT, RIOTS, DOWRY DEATHS, TOTAL IPC CRIMES
    'ncrb_ipc_district_2001_2012': (
        'https://raw.githubusercontent.com/ankurrastogi09/crime_in_india_data_analysis'
        '/master/01_District_wise_crimes_committed_IPC_2001_2012.csv'
    ),

    # ── IPC Crimes — District-wise 2013 ──────────────────────────────────
    # Same schema, continuation year
    'ncrb_ipc_district_2013': (
        'https://raw.githubusercontent.com/Manoj-M-97/Analysing-Patterns-of-Crime-in-India'
        '/master/dataset/2013.csv'
    ),

    # ── IPC Crimes — District-wise 2014 ──────────────────────────────────
    'ncrb_ipc_district_2014': (
        'https://raw.githubusercontent.com/BhanuPratapSunda1/Crime-Data-Analysis'
        '/main/01_District_wise_crimes_committed_IPC_2001_2012_.csv'
    ),

    # ── State-wise crime summary ──────────────────────────────────────────
    'ncrb_crime_statewise': (
        'https://raw.githubusercontent.com/navneet-nmk/Indian-Crime-Data'
        '/master/crime_data.csv'
    ),

}

print('\n' + '═' * 60)
print('PART B — NCRB datasets via GitHub mirrors')
print('═' * 60)

for tbl, url in GITHUB_DATASETS.items():
    print(f'\n📥 {tbl}')
    print(f'   {url}')
    try:
        r = requests.get(url, timeout=30, headers=HEADERS)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text), on_bad_lines='skip')
        df.dropna(how='all', inplace=True)
        print(f'  ✅ {df.shape[0]:,} rows × {df.shape[1]} cols')
        print(f'  Cols: {list(df.columns[:6])}...')
        display(df.head(3))
        save_table(df, tbl)
    except Exception as e:
        print(f'  ⚠️  {e}')
        print(f'  ℹ️  Download manually and use load_uploaded_file("{tbl}.csv", "{tbl}")')


# ═══════════════════════════════════════════════════════════════════════════
# PART C — Manual download guide for datasets not available programmatically
# ═══════════════════════════════════════════════════════════════════════════

print('\n' + '═' * 60)
print('PART C — Manual downloads (Kaggle / NCRB portal / NJDG)')
print('═' * 60)
print("""
These datasets require a free account or manual download.
After downloading, upload via: Catalog → Volumes → legal_files → Upload
Then run load_uploaded_file() from Cell 12.

┌─────────────────────────────────────────────────────────────────────────┐
│ 1. NCRB Crime in India 2001–2022 (state+district, 40+ crime categories) │
│    URL  : kaggle.com/datasets/rajanand/crime-in-india                   │
│    Load : load_uploaded_file('01_District_wise_crimes_committed_        │
│                               IPC_2013.csv', 'ncrb_ipc_2013')          │
├─────────────────────────────────────────────────────────────────────────┤
│ 2. NCRB Prison Statistics India 2022 (PSI)                              │
│    URL  : ncrb.gov.in → PSI 2022 → Data Tables                         │
│    Load : load_uploaded_file('PSI_Table1.xlsx', 'ncrb_prison_2022',    │
│                               fmt='excel', skip_rows=3)                 │
├─────────────────────────────────────────────────────────────────────────┤
│ 3. NCRB Crimes Against Women 2022 (district-wise)                       │
│    URL  : kaggle.com/datasets/ananya0001/                               │
│                crimes-against-women-in-india-2022                       │
│    Load : load_uploaded_file('crimes_women.csv', 'ncrb_crime_women')   │
├─────────────────────────────────────────────────────────────────────────┤
│ 4. NJDG State-wise Court Pendency                                       │
│    URL  : njdg.ecourts.gov.in → National Summary → Download CSV        │
│    Load : load_uploaded_file('njdg_summary.csv', 'njdg_pendency')      │
├─────────────────────────────────────────────────────────────────────────┤
│ 5. Dept of Justice — Judge Vacancies & Court Infrastructure             │
│    URL  : doj.gov.in → Statistics → Vacancy Position                   │
│    Load : load_uploaded_file('judge_vacancies.xlsx',                    │
│                               'doj_judge_vacancies', fmt='excel')       │
└─────────────────────────────────────────────────────────────────────────┘
""")

════════════════════════════════════════════════════════════
PART A — data.gov.in confirmed APIs
════════════════════════════════════════════════════════════

📥 ncrb_ipc_crime_heads
  ncrb_ipc_crime_heads: 36/36
  ✅ ncrb_ipc_crime_heads: 36 records


sl__no,name_of_the_state___ut,pending_cases_in_district___subordinat_courts_as_on_31_12_2015
1,Uttar Pradesh,5574490
2,Maharashtra,2994074
3,West Bengal,2618813


  💾 main.india_legal.ncrb_ipc_crime_heads  (36 rows × 3 cols)

📥 pending_court_cases
  ⚠️  offset 500: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

  ✅ pending_court_cases: 500 records


country,state,city,station,last_update,latitude,longitude,pollutant_id,min_value,max_value,avg_value
India,Andhra_Pradesh,Anantapur,"Gulzarpet, Anantapur - APPCB",06-03-2026 14:00:00,14.675886,77.593027,SO2,10,18,11
India,Andhra_Pradesh,Chittoor,"Gangineni Cheruvu, Chittoor - APPCB",06-03-2026 14:00:00,13.204880,79.097889,SO2,10,11,11
India,Andhra_Pradesh,Kadapa,"Yerramukkapalli, Kadapa - APPCB",06-03-2026 14:00:00,14.465052,78.824187,NO2,NA,NA,NA


  💾 main.india_legal.pending_court_cases  (500 rows × 11 cols)

════════════════════════════════════════════════════════════
PART B — NCRB datasets via GitHub mirrors
════════════════════════════════════════════════════════════

📥 ncrb_ipc_district_2001_2012
   https://raw.githubusercontent.com/ankurrastogi09/crime_in_india_data_analysis/master/01_District_wise_crimes_committed_IPC_2001_2012.csv
  ✅ 9,017 rows × 33 cols
  Cols: ['STATE/UT', 'DISTRICT', 'YEAR', 'MURDER', 'ATTEMPT TO MURDER', 'CULPABLE HOMICIDE NOT AMOUNTING TO MURDER']...


...

  💾 main.india_legal.ncrb_ipc_district_2001_2012  (9017 rows × 33 cols)

📥 ncrb_ipc_district_2013
   https://raw.githubusercontent.com/Manoj-M-97/Analysing-Patterns-of-Crime-in-India/master/dataset/2013.csv
  ⚠️  404 Client Error: Not Found for url: https://raw.githubusercontent.com/Manoj-M-97/Analysing-Patterns-of-Crime-in-India/master/dataset/2013.csv
  ℹ️  Download manually and use load_uploaded_file("ncrb_ipc_district_2013.csv", "ncrb_ipc_district_2013")

📥 ncrb_ipc_district_2014
   https://raw.githubusercontent.com/BhanuPratapSunda1/Crime-Data-Analysis/main/01_District_wise_crimes_committed_IPC_2001_2012_.csv
  ✅ 9,017 rows × 33 cols
  Cols: ['STATE.UT', 'DISTRICT', 'YEAR', 'MURDER', 'ATTEMPT.TO.MURDER', 'CULPABLE.HOMICIDE.NOT.AMOUNTING.TO.MURDER']...


STATE.UT,DISTRICT,YEAR,MURDER,ATTEMPT.TO.MURDER,CULPABLE.HOMICIDE.NOT.AMOUNTING.TO.MURDER,RAPE,CUSTODIAL.RAPE,OTHER.RAPE,KIDNAPPING...ABDUCTION,KIDNAPPING.AND.ABDUCTION.OF.WOMEN.AND.GIRLS,KIDNAPPING.AND.ABDUCTION.OF.OTHERS,DACOITY,PREPARATION.AND.ASSEMBLY.FOR.DACOITY,ROBBERY,BURGLARY,THEFT,AUTO.THEFT,OTHER.THEFT,RIOTS,CRIMINAL.BREACH.OF.TRUST,CHEATING,COUNTERFIETING,ARSON,HURT.GREVIOUS.HURT,DOWRY.DEATHS,ASSAULT.ON.WOMEN.WITH.INTENT.TO.OUTRAGE.HER.MODESTY,INSULT.TO.MODESTY.OF.WOMEN,CRUELTY.BY.HUSBAND.OR.HIS.RELATIVES,IMPORTATION.OF.GIRLS.FROM.FOREIGN.COUNTRIES,CAUSING.DEATH.BY.NEGLIGENCE,OTHER.IPC.CRIMES,TOTAL.IPC.CRIMES
ANDHRA PRADESH,ADILABAD,2001,101,60,17,50,0,50,46,30,16,9,0,41,198,199,22,177,78,16,104,1,30,1131,16,149,34,175,0,181,1518,4154
ANDHRA PRADESH,ANANTAPUR,2001,151,125,1,23,0,23,53,30,23,8,0,16,191,366,57,309,168,11,65,8,69,1543,7,118,24,154,0,270,754,4125
ANDHRA PRADESH,CHITTOOR,2001,101,57,2,27,0,27,59,34,25,4,0,14,237,723,164,559,156,33,209,9,38,2088,14,112,83,186,0,404,1262,5818


  💾 main.india_legal.ncrb_ipc_district_2014  (9017 rows × 33 cols)

📥 ncrb_crime_statewise
   https://raw.githubusercontent.com/navneet-nmk/Indian-Crime-Data/master/crime_data.csv
  ✅ 838 rows × 91 cols
  Cols: ['States/UTs', 'District', 'Year', 'Murder', 'Attempt to commit Murder', 'Culpable Homicide not amounting to Murder']...


States/UTs,District,Year,Murder,Attempt to commit Murder,Culpable Homicide not amounting to Murder,Attempt to commit Culpable Homicide,Rape,Custodial Rape,Custodial_Gang Rape,Custodial_Other Rape,Rape other than Custodial,Rape_Gang Rape,Rape_Others,Attempt to commit Rape,Kidnapping & Abduction_Total,Kidnapping & Abduction,Kidnapping & Abduction in order to Murder,Kidnapping for Ransom,Kidnapping & Abduction of Women to compel her for marriage,Other Kidnapping,Dacoity,Dacoity with Murder,Other Dacoity,Making Preparation and Assembly for committing Dacoity,Robbery,Criminal Trespass/Burglary,Criminal Trespass or Burglary,House Trespass & House Breaking,Theft,Auto Theft,Other Thefts,Unlawful Assembly,Riots,Riots_Communal,Riots_Industrial,Riots_Political,Riots_Caste Conflict,Riots_SC/STs Vs Non-SCs/STs,Riots_Other Caste Conflict,Riots_Agrarian,Riots_Students,Riots_Sectarian,Riots_Others,Criminal Breach of Trust,Cheating,Forgery,Counterfeiting,Counterfeit Offences related to Counterfeit Coin,Counterfeiting Government Stamp,Counterfeit currency & Bank notes,Counterfeiting currency notes/Bank notes,Using forged or counterfeiting currency/Bank notes,Possession of forged or counterfeiting currency/Bank notes,Making or Possessing materials for forged currency/Bank notes,Making or Using documents resembling currency,Arson,Grievous Hurt,Hurt,Acid attack,Attempt to Acid Attack,Dowry Deaths,Assault on Women with intent to outrage her Modesty,Sexual Harassment,Assault or use of criminal force to women with intent to Disrobe,Voyeurism,Stalking,Other Assault on Women,Insult to the Modesty of Women,At Office premises,Other places related to work,In Public Transport system,"Places other than 231, 232 & 233",Cruelty by Husband or his Relatives,Importation of Girls from Foreign Country,Causing Death by Negligence,Deaths due to negligent driving/act,Deaths due to Other Causes,Offences against State,Sedition,Other offences against State,Offences promoting enmity between different groups,Promoting enmity between different groups,"Imputation, assertions prejudicial to national integration",Extortion,Disclosure of Identity of Victims,Incidence of Rash Driving,HumanTrafficking,Unnatural Offence,Other IPC crimes,Total Cognizable IPC crimes
Andhra Pradesh,Anantapur,2014,134,171,8,0,35,0,0,0,35,0,35,1,125,0,0,0,88,37,6,0,6,0,30,415,315,100,753,240,513,0,214,0,0,0,0,0,0,0,0,0,214,13,296,0,10,0,0,10,10,0,0,0,0,12,25,24,1,0,25,436,82,34,4,80,236,26,0,0,0,26,165,0,638,638,0,0,0,0,0,0,0,0,0,1038,0,0,3800,8376
Andhra Pradesh,Chittoor,2014,84,170,2,0,32,0,0,0,32,1,31,0,38,4,0,3,28,3,12,0,12,0,22,195,154,41,528,192,336,0,134,0,0,0,0,0,0,0,0,0,134,7,207,0,5,0,0,5,0,5,0,0,0,23,18,18,0,0,17,135,0,0,0,0,135,94,0,0,0,94,278,0,538,538,0,0,0,0,0,0,0,19,0,249,0,0,2567,5374
Andhra Pradesh,Cuddapah,2014,80,162,1,0,28,0,0,0,28,0,28,4,27,0,0,0,11,16,3,0,3,0,16,144,144,0,638,193,445,0,104,0,0,104,0,0,0,0,0,0,0,86,163,0,5,0,0,5,5,0,0,0,0,0,39,39,0,0,16,215,212,0,0,0,3,12,1,11,0,0,91,0,417,417,0,0,0,0,0,0,0,0,0,948,0,0,2604,5803


  💾 main.india_legal.ncrb_crime_statewise  (838 rows × 91 cols)

════════════════════════════════════════════════════════════
PART C — Manual downloads (Kaggle / NCRB portal / NJDG)
════════════════════════════════════════════════════════════

These datasets require a free account or manual download.
After downloading, upload via: Catalog → Volumes → legal_files → Upload
Then run load_uploaded_file() from Cell 12.

┌─────────────────────────────────────────────────────────────────────────┐
│ 1. NCRB Crime in India 2001–2022 (state+district, 40+ crime categories) │
│    URL  : kaggle.com/datasets/rajanand/crime-in-india                   │
│    Load : load_uploaded_file('01_District_wise_crimes_committed_        │
│                               IPC_2013.csv', 'ncrb_ipc_2013')          │
├─────────────────────────────────────────────────────────────────────────┤
│ 2. NCRB Prison Statistics India 2022 (PSI)                              │
│    URL  : ncrb.gov.in → PSI 2022 → Data Tables  

## 3b. data.gov.in — Ingest selected datasets
Edit `INGEST_TARGETS` with resource IDs from 3a.

In [0]:
# ── Cell 3b: data.gov.in — Ingest all records from confirmed resources ────────
#
# Reads from CONFIRMED_APIS defined in cell 3a.
# Does NOT depend on df_catalog having a 'title' column.
# Paginates 100 records/request until all rows are fetched.
# ─────────────────────────────────────────────────────────────────────────────

# These are the only two confirmed working API resources.
# Add more UUIDs here if you discover new ones via data.gov.in.
INGEST_TARGETS = {
    'b031d51c-e7db-4ef6-a47a-08ee97fd9f62': 'ncrb_ipc_crime_heads',
    '3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69': 'pending_court_cases',
    # Paste additional UUIDs here:
    # 'PASTE-UUID': 'table_name',
}

def fetch_all_datagov(resource_id, label, max_records=50000):
    """Fetch ALL rows from a data.gov.in resource, paginating 100 at a time."""
    base  = f'https://api.data.gov.in/resource/{resource_id}'
    recs, offset = [], 0

    # First probe: get total count and validate the resource exists
    try:
        r = requests.get(base, params={
            'api-key': DATAGOV_API_KEY, 'format': 'json',
            'limit': 1, 'offset': 0
        }, timeout=15, headers=HEADERS)
        r.raise_for_status()
        total_available = int(r.json().get('total', 0))
        if total_available == 0:
            print(f'  ⚠️  {label}: 0 records available — skipping')
            return pd.DataFrame()
        print(f'  {label}: {total_available:,} records available, fetching...')
    except Exception as e:
        print(f'  ⚠️  {label}: probe failed — {e}')
        return pd.DataFrame()

    # Paginate
    while offset < min(max_records, total_available):
        try:
            r = requests.get(base, params={
                'api-key': DATAGOV_API_KEY, 'format': 'json',
                'limit': 100, 'offset': offset
            }, timeout=30, headers=HEADERS)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f'\n  ⚠️  offset {offset}: {e}')
            break

        batch = data.get('records', [])
        if not batch:
            break

        recs.extend(batch)
        offset += 100
        print(f'  {label}: {len(recs):,}/{min(total_available, max_records):,}', end='\r')
        time.sleep(0.25)

    print(f'\n  ✅ {label}: {len(recs):,} records fetched')
    return pd.DataFrame(recs) if recs else pd.DataFrame()


# ── Run ingestion ─────────────────────────────────────────────────────────────
print(f'📥 Ingesting {len(INGEST_TARGETS)} dataset(s) from data.gov.in...\n')

ingested = {}
for rid, tbl in INGEST_TARGETS.items():
    print(f'── {tbl}  [{rid}]')
    df = fetch_all_datagov(rid, tbl)
    if not df.empty:
        display(df.head(3))
        save_table(df, tbl)
        ingested[tbl] = df.shape
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
print('─' * 50)
print(f'✅ Done. {len(ingested)}/{len(INGEST_TARGETS)} tables saved:\n')
for tbl, shape in ingested.items():
    print(f'   {CATALOG}.{SCHEMA}.{tbl}  →  {shape[0]:,} rows × {shape[1]} cols')

if len(ingested) < len(INGEST_TARGETS):
    missed = [t for t in INGEST_TARGETS.values() if t not in ingested]
    print(f'\n⚠️  Skipped (no data): {missed}')
    print('   These UUIDs may not have an active API on data.gov.in.')
    print('   Use Part B / Part C from cell 3a to get data from GitHub/Kaggle instead.')

📥 Ingesting 2 dataset(s) from data.gov.in...

── ncrb_ipc_crime_heads  [b031d51c-e7db-4ef6-a47a-08ee97fd9f62]
  ncrb_ipc_crime_heads: 36 records available, fetching...
  ncrb_ipc_crime_heads: 36/36
  ✅ ncrb_ipc_crime_heads: 36 records fetched


sl__no,name_of_the_state___ut,pending_cases_in_district___subordinat_courts_as_on_31_12_2015
1,Uttar Pradesh,5574490
2,Maharashtra,2994074
3,West Bengal,2618813


  💾 main.india_legal.ncrb_ipc_crime_heads  (36 rows × 3 cols)

── pending_court_cases  [3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69]
  pending_court_cases: 3,329 records available, fetching...
  pending_court_cases: 1,200/3,329
  ⚠️  offset 1200: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

  ✅ pending_court_cases: 1,200 records fetched


country,state,city,station,last_update,latitude,longitude,pollutant_id,min_value,max_value,avg_value
India,Andhra_Pradesh,Anantapur,"Gulzarpet, Anantapur - APPCB",06-03-2026 14:00:00,14.675886,77.593027,SO2,10,18,11
India,Andhra_Pradesh,Chittoor,"Gangineni Cheruvu, Chittoor - APPCB",06-03-2026 14:00:00,13.204880,79.097889,SO2,10,11,11
India,Andhra_Pradesh,Kadapa,"Yerramukkapalli, Kadapa - APPCB",06-03-2026 14:00:00,14.465052,78.824187,NO2,NA,NA,NA


  💾 main.india_legal.pending_court_cases  (1200 rows × 11 cols)

──────────────────────────────────────────────────
✅ Done. 2/2 tables saved:

   main.india_legal.ncrb_ipc_crime_heads  →  36 rows × 3 cols
   main.india_legal.pending_court_cases  →  1,200 rows × 11 cols


## 4a. DAKSH India — Portal API
Tries DAKSH portal JSON endpoints.

In [0]:
DAKSH_ENDPOINTS = {
    'daksh_pendency_summary': 'https://database.dakshindia.org/api/pendency-summary',
    'daksh_disposal_summary': 'https://database.dakshindia.org/api/disposal-summary',
    'daksh_court_list':       'https://database.dakshindia.org/api/courts',
}
for tbl, url in DAKSH_ENDPOINTS.items():
    print(f'📥 {tbl}')
    try:
        r  = requests.get(url, timeout=20, headers={**HEADERS, 'Accept': 'application/json'})
        r.raise_for_status()
        df = pd.DataFrame(r.json() if isinstance(r.json(), list) else [r.json()])
        print(f'  ✅ {df.shape[0]} rows')
        save_table(df, tbl)
    except Exception as e:
        print(f'  ⚠️  {e}')

📥 daksh_pendency_summary
  ⚠️  404 Client Error: Not Found for url: https://database.dakshindia.org/api/pendency-summary
📥 daksh_disposal_summary
  ⚠️  404 Client Error: Not Found for url: https://database.dakshindia.org/api/disposal-summary
📥 daksh_court_list
  ⚠️  404 Client Error: Not Found for url: https://database.dakshindia.org/api/courts


## 4b. DAKSH India — HTML scrape
Falls back to scraping the DAKSH portal page for summary tables.

In [0]:
print('📥 Scraping DAKSH portal...')
try:
    r    = requests.get('https://database.dakshindia.org/', timeout=30, headers=HEADERS)
    soup = BeautifulSoup(r.content, 'html.parser')
    frames = []
    for t in soup.find_all('table'):
        try:
            df_t = pd.read_html(str(t))[0]
            if df_t.shape[1] > 1:
                frames.append(df_t)
        except Exception:
            pass
    if frames:
        df_daksh = pd.concat(frames, ignore_index=True)
        print(f'  ✅ {df_daksh.shape[0]} rows scraped')
        save_table(df_daksh, 'daksh_portal_summary')
    else:
        print('  ⚠️  No tables found — portal is JS-rendered')
        print('  👉 https://database.dakshindia.org → Resources → download CSVs')
        print('     then upload to Volume and use Section 12')
except Exception as e:
    print(f'  ⚠️  {e}')

📥 Scraping DAKSH portal...
  ⚠️  No tables found — portal is JS-rendered
  👉 https://database.dakshindia.org → Resources → download CSVs
     then upload to Volume and use Section 12


## 5. PRS India — MP Legislative Activity
GitHub-hosted semicolon-delimited CSV.

In [0]:
MP_URL = ('https://raw.githubusercontent.com/Vonter/india-representatives-activity'
          '/main/csv/Lok%20Sabha/18th.csv')
print('📥 Fetching MP Activity (18th Lok Sabha)...')
try:
    r  = requests.get(MP_URL, timeout=30, headers=HEADERS)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), sep=';', quotechar='"', on_bad_lines='skip')
    df.dropna(axis=1, how='all', inplace=True)
    if df.shape[1] <= 3 and ';' in str(df.iloc[0, 0]):
        rows = [line.strip().strip('"').split('";"') for line in r.text.splitlines()]
        df   = pd.DataFrame(rows[1:], columns=rows[0])
        df.dropna(axis=1, how='all', inplace=True)
    print(f'  ✅ {df.shape[0]} MPs × {df.shape[1]} cols')
    display(df[['Name','Party','State','Attendance','Debates','Questions']].head(5))
    save_table(df, 'mp_activity_loksabha18')
except Exception as e:
    print(f'  ⚠️  {e}')

📥 Fetching MP Activity (18th Lok Sabha)...
  ✅ 544 MPs × 84 cols


Name,Party,State,Attendance,Debates,Questions
Aashtikar Patil Nagesh Bapurao,Shiv Sena (Uddhav Balasaheb Thackeray),Maharashtra,60.00%,0,43
Abdul Rashid Sheikh,Independent,Jammu And Kashmir,0.00%,0,0
Abhay Kumar Sinha,Rashtriya Janata Dal,Bihar,80.00%,11,11
Abhijit Gangopadhyay,Bharatiya Janata Party,West Bengal,85.00%,2,0
Abhishek Banerjee,All India Trinamool Congress,West Bengal,35.00%,1,19


  💾 main.india_legal.mp_activity_loksabha18  (544 rows × 84 cols)


## 6a. PRS Bills — Open session
Opens a requests.Session and fetches the Drupal form_build_id needed for AJAX POSTs.

In [0]:
BASE_URL = 'https://prsindia.org'
bills_session = requests.Session()
bills_session.headers.update({
    **HEADERS,
    'Accept':           'application/json, text/javascript, */*; q=0.01',
    'X-Requested-With': 'XMLHttpRequest',
    'Content-Type':     'application/x-www-form-urlencoded; charset=UTF-8',
})
bills_form_build_id = ''
try:
    page_r = bills_session.get(f'{BASE_URL}/billtrack', timeout=20)
    soup   = BeautifulSoup(page_r.content, 'html.parser')
    inp    = soup.find('input', {'name': 'form_build_id'})
    if inp:
        bills_form_build_id = inp.get('value', '')
    print(f'✅ Bills session ready')
    print(f'   form_build_id: {bills_form_build_id[:40]}...')
except Exception as e:
    print(f'⚠️  {e}')

✅ Bills session ready
   form_build_id: ...


## 6b. PRS Bills — AJAX paginated fetch
POSTs to /views/ajax per page replicating browser behaviour.

In [0]:
bills_session.headers.update({'Referer': f'{BASE_URL}/billtrack', 'Origin': BASE_URL})
all_bills = []
MAX_PAGES = 30

for page_num in range(MAX_PAGES):
    payload = {
        'view_name':       'bills_track',
        'view_display_id': 'page_1',
        'view_args':       '',
        'view_path':       '/billtrack',
        'view_base_path':  'billtrack',
        'view_dom_id':     '1',
        'pager_element':   '0',
        'page':            str(page_num),
        'form_build_id':   bills_form_build_id,
    }
    try:
        r        = bills_session.post(f'{BASE_URL}/views/ajax', data=payload, timeout=20)
        commands = r.json()
    except Exception as e:
        print(f'  ⚠️  Page {page_num}: {e}')
        break
    html_chunk = next(
        (c.get('data','') for c in commands if isinstance(c, dict) and c.get('command') == 'insert'), ''
    )
    if not html_chunk:
        print(f'  No content at page {page_num} — done')
        break
    rows       = BeautifulSoup(html_chunk, 'html.parser').select('tbody tr')
    page_bills = []
    for row in rows:
        cols = row.find_all('td')
        a    = row.find('a')
        if not cols:
            continue
        page_bills.append({
            'title':    a.get_text(strip=True) if a else cols[0].get_text(strip=True),
            'url':      BASE_URL + a['href'] if a and a.get('href') else '',
            'ministry': cols[1].get_text(strip=True) if len(cols) > 1 else '',
            'status':   cols[2].get_text(strip=True) if len(cols) > 2 else '',
            'session':  cols[3].get_text(strip=True) if len(cols) > 3 else '',
            'house':    cols[4].get_text(strip=True) if len(cols) > 4 else '',
        })
    if not page_bills:
        print(f'  No rows at page {page_num} — done')
        break
    all_bills.extend(page_bills)
    print(f'  Page {page_num:2d}: +{len(page_bills)} bills  (total: {len(all_bills)})', end='\r')
    time.sleep(0.4)

print(f'\n✅ Total bills fetched: {len(all_bills)}')

  ⚠️  Page 0: Expecting value: line 1 column 1 (char 0)

✅ Total bills fetched: 0


## 6c. PRS Bills — Save or fallback
Saves bills to Delta, or tries sansad.in if AJAX returned nothing.

In [0]:
if all_bills:
    df_bills = pd.DataFrame(all_bills)
    display(df_bills.head(10))
    save_table(df_bills, 'prs_bills')
else:
    print('⚠️  AJAX returned 0 bills. Trying sansad.in fallback...')
    try:
        r    = requests.get('https://sansad.in/ls/legislation/bills', headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.content, 'html.parser')
        bills = []
        for row in soup.select('tr'):
            cols = row.find_all('td')
            a    = row.find('a')
            if len(cols) >= 2:
                bills.append({
                    'title':   a.get_text(strip=True) if a else cols[0].get_text(strip=True),
                    'url':     'https://sansad.in' + a['href'] if a and a.get('href') else '',
                    'bill_no': cols[1].get_text(strip=True) if len(cols) > 1 else '',
                    'year':    cols[2].get_text(strip=True) if len(cols) > 2 else '',
                    'status':  cols[3].get_text(strip=True) if len(cols) > 3 else '',
                    'source':  'sansad.in',
                })
        df_bills = pd.DataFrame(bills)
        print(f'  ✅ sansad.in: {len(bills)} bills')
        if not df_bills.empty:
            display(df_bills.head(10))
            save_table(df_bills, 'prs_bills')
    except Exception as e:
        print(f'  ⚠️  {e}')
        print('  Manual: https://prsindia.org/billtrack → Save As → CSV')

⚠️  AJAX returned 0 bills. Trying sansad.in fallback...
  ✅ sansad.in: 0 bills


## 7a. PRS Acts — Open session
Same AJAX pattern as bills but targeting the acts_listing view.

In [0]:
acts_session = requests.Session()
acts_session.headers.update({
    **HEADERS,
    'Accept':           'application/json, text/javascript, */*; q=0.01',
    'X-Requested-With': 'XMLHttpRequest',
    'Content-Type':     'application/x-www-form-urlencoded; charset=UTF-8',
    'Referer':          f'{BASE_URL}/acts/parliament',
    'Origin':           BASE_URL,
})
acts_form_build_id = ''
try:
    page_r = acts_session.get(f'{BASE_URL}/acts/parliament', timeout=20)
    inp    = BeautifulSoup(page_r.content, 'html.parser').find('input', {'name': 'form_build_id'})
    if inp:
        acts_form_build_id = inp.get('value', '')
    print(f'✅ Acts session ready')
    print(f'   form_build_id: {acts_form_build_id[:40]}...')
except Exception as e:
    print(f'⚠️  {e}')

✅ Acts session ready
   form_build_id: ...


## 7b. PRS Acts — AJAX paginated fetch

In [0]:
all_acts  = []
MAX_PAGES = 50

for page_num in range(MAX_PAGES):
    payload = {
        'view_name':       'acts_listing',
        'view_display_id': 'page_1',
        'view_args':       '',
        'view_path':       '/acts/parliament',
        'view_base_path':  'acts/parliament',
        'view_dom_id':     '1',
        'pager_element':   '0',
        'page':            str(page_num),
        'form_build_id':   acts_form_build_id,
    }
    try:
        r        = acts_session.post(f'{BASE_URL}/views/ajax', data=payload, timeout=20)
        commands = r.json()
    except Exception as e:
        print(f'  ⚠️  Page {page_num}: {e}')
        break
    html_chunk = next(
        (c.get('data','') for c in commands if isinstance(c, dict) and c.get('command') == 'insert'), ''
    )
    if not html_chunk:
        print(f'  No content at page {page_num} — done')
        break
    rows      = BeautifulSoup(html_chunk, 'html.parser').select('tbody tr')
    page_acts = []
    for row in rows:
        cols = row.find_all('td')
        a    = row.find('a')
        if not cols:
            continue
        page_acts.append({
            'act_title': a.get_text(strip=True) if a else cols[0].get_text(strip=True),
            'url':       BASE_URL + a['href'] if a and a.get('href','').startswith('/') else '',
            'year':      cols[1].get_text(strip=True) if len(cols) > 1 else '',
            'ministry':  cols[2].get_text(strip=True) if len(cols) > 2 else '',
        })
    if not page_acts:
        print(f'  No rows at page {page_num} — done')
        break
    all_acts.extend(page_acts)
    print(f'  Page {page_num:2d}: +{len(page_acts)} acts  (total: {len(all_acts)})', end='\r')
    time.sleep(0.4)

print(f'\n✅ Total acts fetched: {len(all_acts)}')

  ⚠️  Page 0: Expecting value: line 1 column 1 (char 0)

✅ Total acts fetched: 0


## 7c. PRS Acts — Save

In [0]:
if all_acts:
    df_acts = pd.DataFrame(all_acts)
    display(df_acts.head(10))
    save_table(df_acts, 'prs_acts')
else:
    print('⚠️  No acts fetched.')
    print('   Check Network tab on https://prsindia.org/acts/parliament')
    print('   and confirm view_name / view_display_id in cell 7b payload.')

⚠️  No acts fetched.
   Check Network tab on https://prsindia.org/acts/parliament
   and confirm view_name / view_display_id in cell 7b payload.


## 8. Legal PDFs — BNS / Constitution / BNSS / BSA
Downloads official PDFs from indiacode.nic.in into the Unity Catalog Volume.
**Note:** Uses Volume path — NOT /dbfs/FileStore (read-only on Free Edition).

In [0]:
# ── Cell 8: Download official legal PDFs to Unity Catalog Volume ─────────────
#
# WHY THE OLD URLS FAILED:
#   indiacode.nic.in /bitstream/.../bns_2023.pdf  →  redirects to a login/error
#   page (~18 KB HTML), not the actual PDF.
#
# FIXED URLS:
#   BNS  & BNSS → MHA (Ministry of Home Affairs) direct Gazette PDFs
#   BSA  & Constitution → indiacode.nic.in bitstream (different handle numbers)
#
# SIZE CHECK: a real PDF will be > 500 KB. Anything < 100 KB is an error page.
# ─────────────────────────────────────────────────────────────────────────────

LEGAL_PDFS = {
    'bns_2023': {
        'url':  'https://www.mha.gov.in/sites/default/files/250883_english_01042024.pdf',
        'desc': 'Bharatiya Nyaya Sanhita 2023 — replaces IPC (MHA Gazette)',
        'min_bytes': 500_000,   # real PDF is ~1.5 MB
    },
    'bnss_2023': {
        'url':  'https://www.mha.gov.in/sites/default/files/2024-04/250884_2_english_01042024.pdf',
        'desc': 'Bharatiya Nagarik Suraksha Sanhita 2023 — replaces CrPC (MHA Gazette)',
        'min_bytes': 500_000,
    },
    'bsa_2023': {
        # Primary: MHA Gazette (same source as BNS/BNSS, confirmed pattern)
        # Fallback 1: Tripura HC hosts a clean copy from NIC
        # Fallback 2: Indian Railways NIC upload
        'url':  'https://www.mha.gov.in/sites/default/files/2024-04/250882_english_01042024_0.pdf',
        'fallback_urls': [
            'https://thc.nic.in/Central%20Governmental%20Acts/Bharatiya%20Sakshya%20Adhiniyam,%202023.pdf',
            'https://ncr.indianrailways.gov.in/uploads/files/1748345531944-Bharatiya%20Sakshya%20Adhiniyam-2023(English).pdf',
        ],
        'desc': 'Bharatiya Sakshya Adhiniyam 2023 — replaces Evidence Act (MHA Gazette)',
        'min_bytes': 200_000,
    },
    'constitution_of_india': {
        'url':  'https://www.indiacode.nic.in/bitstream/123456789/19151/1/constitution_of_india.pdf',
        'desc': 'Constitution of India (as amended)',
        'min_bytes': 1_000_000,  # ~3 MB
    },
}

pdf_paths = {}

for name, meta in LEGAL_PDFS.items():
    dest = f'{PDF_DIR}/{name}.pdf'

    # Check if already downloaded and valid
    if os.path.exists(dest):
        size = os.path.getsize(dest)
        if size >= meta['min_bytes']:
            print(f'  ✅ Already valid: {name}.pdf  ({size:,} bytes)')
            pdf_paths[name] = dest
            continue
        else:
            print(f'  ⚠️  Existing {name}.pdf is too small ({size:,} bytes) — re-downloading')
            os.remove(dest)

    # Build list of URLs to try: primary + any fallbacks
    urls_to_try = [meta['url']] + meta.get('fallback_urls', [])

    saved = False
    for attempt, url in enumerate(urls_to_try):
        label = 'primary' if attempt == 0 else f'fallback {attempt}'
        print(f'  📥 {name}.pdf  [{label}]')
        print(f'     {meta["desc"]}')
        print(f'     {url}')

        try:
            r = requests.get(
                url,
                timeout=120,
                headers={**HEADERS, 'Accept': 'application/pdf,*/*'},
                stream=True,
                allow_redirects=True,
            )
            r.raise_for_status()

            content_type = r.headers.get('Content-Type', '')
            if 'html' in content_type.lower():
                print(f'     ⚠️  Got HTML ({content_type}) — trying next URL')
                continue

            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)

            size = os.path.getsize(dest)
            if size < meta['min_bytes']:
                print(f'     ⚠️  Too small ({size:,} bytes) — trying next URL')
                os.remove(dest)
                continue

            pdf_paths[name] = dest
            print(f'     ✅ Saved ({size:,} bytes)')
            saved = True
            break

        except Exception as e:
            print(f'     ⚠️  {e} — trying next URL')

    if not saved:
        print(f'  ❌ All URLs failed for {name}.pdf')
        print(f'     Manual download options:')
        for url in urls_to_try:
            print(f'       {url}')
        print(f'     Upload to Volume and rename to {name}.pdf')

    print()

# ── Summary ───────────────────────────────────────────────────────────────────
print('─' * 60)
print(f'✅ {len(pdf_paths)}/{len(LEGAL_PDFS)} PDFs ready in {PDF_DIR}')
for name, path in pdf_paths.items():
    print(f'   {name}.pdf  →  {os.path.getsize(path):,} bytes')

if len(pdf_paths) < len(LEGAL_PDFS):
    missing = [n for n in LEGAL_PDFS if n not in pdf_paths]
    print(f'\n⚠️  Missing: {missing}')
    print('   For each missing PDF:')
    print('   1. Open the URL above in your browser → Save As PDF')
    print('   2. Catalog → Volumes → legal_files → Upload')
    print('   3. Rename to <name>.pdf in the volume')
    print('   4. Cell 9b will find it via pdf_paths dict — re-run this cell after uploading')
    # Still register manually uploaded files if they exist
    for name in missing:
        manual_path = f'{PDF_DIR}/{name}.pdf'
        if os.path.exists(manual_path) and os.path.getsize(manual_path) > 10_000:
            pdf_paths[name] = manual_path
            print(f'   Found manually uploaded: {name}.pdf ✅')

  ✅ Already valid: bns_2023.pdf  (1,324,200 bytes)
  ✅ Already valid: bnss_2023.pdf  (2,033,181 bytes)
  📥 bsa_2023.pdf  [primary]
     Bharatiya Sakshya Adhiniyam 2023 — replaces Evidence Act (MHA Gazette)
     https://www.mha.gov.in/sites/default/files/2024-04/250882_english_01042024_0.pdf
     ✅ Saved (670,370 bytes)

  ✅ Already valid: constitution_of_india.pdf  (2,712,149 bytes)
────────────────────────────────────────────────────────────
✅ 4/4 PDFs ready in /Volumes/main/india_legal/legal_files/pdfs
   bns_2023.pdf  →  1,324,200 bytes
   bnss_2023.pdf  →  2,033,181 bytes
   bsa_2023.pdf  →  670,370 bytes
   constitution_of_india.pdf  →  2,712,149 bytes


## 9a. BNS structured CSV — GitHub mirror
Tries known mirrors before falling back to PDF extraction.

In [0]:
BNS_CSV_MIRRORS = [
    'https://raw.githubusercontent.com/OpenNyAI/Opennyai/main/datasets/bns_sections.csv',
    'https://raw.githubusercontent.com/nandr39/bns-dataset/main/bns_sections.csv',
]
df_bns = pd.DataFrame()
for url in BNS_CSV_MIRRORS:
    print(f'  Trying: {url}')
    try:
        r = requests.get(url, timeout=30, headers=HEADERS)
        r.raise_for_status()
        df_bns = pd.read_csv(StringIO(r.text))
        if df_bns.shape[0] > 10:
            print(f'  ✅ {df_bns.shape[0]} sections from mirror')
            break
    except Exception as e:
        print(f'  ⚠️  {e}')
        df_bns = pd.DataFrame()

if df_bns.empty:
    print('CSV mirrors unavailable — run cell 9b (PDF extraction)')
else:
    display(df_bns.head(3))

  Trying: https://raw.githubusercontent.com/OpenNyAI/Opennyai/main/datasets/bns_sections.csv
  ⚠️  404 Client Error: Not Found for url: https://raw.githubusercontent.com/OpenNyAI/Opennyai/main/datasets/bns_sections.csv
  Trying: https://raw.githubusercontent.com/nandr39/bns-dataset/main/bns_sections.csv
  ⚠️  404 Client Error: Not Found for url: https://raw.githubusercontent.com/nandr39/bns-dataset/main/bns_sections.csv
CSV mirrors unavailable — run cell 9b (PDF extraction)


## 9b. BNS CSV — PDF fallback
Only runs if 9a found nothing. Extracts sections from the downloaded PDF.

In [ ]:
# ── Cell 9b: BNS PDF → Bilingual Delta table using ai_parse_document + Sarvam-M
#
# ARCHITECTURE (Sovereign AI approach):
#   1. ai_parse_document  → extracts ALL text (Hindi + English) preserving layout
#   2. Keep Devanagari    → don't strip Hindi; it carries legal meaning
#   3. Sarvam-M API       → interprets bilingual BNS text natively (Hindi+English)
#                           OpenAI-compatible, free tier, no GPU needed
#
# Get a free Sarvam API key at: https://dashboard.sarvam.ai
# Docs: https://docs.sarvam.ai
# ─────────────────────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.functions import col, explode, expr, get_json_object, to_json, from_json
from pyspark.sql.types import ArrayType, StructField, StringType, IntegerType, StructType
import os
import requests, json, re, time

# ── USER CONFIG ───────────────────────────────────────────────────────────────
SARVAM_API_KEY = _read_secret("SARVAM_API_KEY", "sarvam_api_key")  # secret or env
SARVAM_API_URL = 'https://api.sarvam.ai/v1/chat/completions'
SARVAM_MODEL   = 'sarvam-m'
# ─────────────────────────────────────────────────────────────────────────────

BNS_PDF_PATH = f'{PDF_DIR}/bns_2023.pdf'


# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Parse PDF with ai_parse_document — keep ALL text including Hindi
# ═══════════════════════════════════════════════════════════════════════════

if 'bns_2023' not in pdf_paths:
    print('⚠️  bns_2023.pdf not found — run Cell 8 first.')
else:
    print(f'📄 Parsing BNS PDF with ai_parse_document...')
    print(f'   {BNS_PDF_PATH}  ({os.path.getsize(BNS_PDF_PATH):,} bytes)')

    raw_df = spark.read.format('binaryFile').load(BNS_PDF_PATH)

    # VARIANT from ai_parse_document breaks col('parsed.*') and some Delta writes (Spark Connect).
    # Persist only JSON STRING, then parse with from_json — no VARIANT downstream.
    parsed_json_df = raw_df.select(
        expr("to_json(ai_parse_document(content, map('version', '2.0')))").alias('parsed_json')
    )
    parsed_json_df.write.format('delta').mode('overwrite') \
             .option('overwriteSchema', 'true') \
             .saveAsTable(f'{CATALOG}.{SCHEMA}.bns_parsed_raw')
    parsed_df = spark.table(f'{CATALOG}.{SCHEMA}.bns_parsed_raw')
    print('  ✅ ai_parse_document complete — persisted as parsed_json (STRING)')

    _elem_struct = StructType([
        StructField('type', StringType(), True),
        StructField('content', StringType(), True),
        StructField('page_number', StringType(), True),  # JSON may omit or use string
    ])
    _doc_schema = StructType([
        StructField('elements', ArrayType(_elem_struct), True),
    ])

    elements_df = (
        parsed_df
        .select(from_json(col('parsed_json'), _doc_schema).alias('doc'))
        .select(explode(col('doc.elements')).alias('elem'))
        .select(
            col('elem.type').alias('element_type'),
            col('elem.content').alias('content'),
            col('elem.page_number').cast('int').alias('page_number'),
        )
        .filter(col('element_type').isin('TEXT', 'PARAGRAPH', 'HEADING'))
        .orderBy('page_number')
    )

    # Save full parsed elements as queryable Delta table
    (
        elements_df.write.format('delta')
        .mode('overwrite').option('overwriteSchema', 'true')
        .saveAsTable(f'{CATALOG}.{SCHEMA}.bns_parsed_elements')
    )
    total_elements = spark.table(f'{CATALOG}.{SCHEMA}.bns_parsed_elements').count()
    print(f'  💾 bns_parsed_elements: {total_elements} elements saved')


    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 2: Reconstruct BNS sections — bilingual (Hindi + English)
    # ═══════════════════════════════════════════════════════════════════════════

    all_rows   = elements_df.collect()
    full_text  = '\n'.join(
        r['content'] for r in all_rows
        if r['content'] and len(r['content'].strip()) > 3
    )
    print(f'\n  Full bilingual text: {len(full_text):,} chars')

    # Section detection: works on both Hindi and English section patterns
    # BNS uses: "103. ..." in English and "धारा 103." in Hindi portions
    section_pat = re.compile(
        r'(?m)^(\d{1,3})\.\s+(.{2,150}?)(?=[\.—–\n])',
    )
    matches   = list(section_pat.finditer(full_text))
    seen_nums = set()
    raw_sections = []

    for i, m in enumerate(matches):
        num   = m.group(1).strip()
        title = m.group(2).strip().rstrip('.—– ')
        if num in seen_nums:
            continue
        body_start = m.end()
        body_end   = matches[i+1].start() if i+1 < len(matches) else body_start + 3000
        body       = re.sub(r'\s+', ' ', full_text[body_start:body_end]).strip()
        if len(body) < 10:
            continue
        seen_nums.add(num)
        raw_sections.append({
            'section_number': num,
            'section_title':  title,
            'section_text':   body[:3000],   # bilingual: Hindi + English
            'source':         'BNS_2023',
        })

    raw_sections.sort(key=lambda x: int(x['section_number'])
                      if x['section_number'].isdigit() else 999)
    print(f'  Extracted {len(raw_sections)} bilingual sections')


    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 3: Enrich sections with Sarvam-M
    #         → English summary + Hindi explanation for each section
    # ═══════════════════════════════════════════════════════════════════════════

    def sarvam_enrich(section_number: str, section_text: str,
                      retries: int = 2) -> dict:
        """
        Call Sarvam-M API to produce:
          - english_summary  : plain-English 2-line summary
          - hindi_explanation: Hindi explanation of the section
          - key_terms        : legal terms (Hindi + English)
        """
        prompt = f"""You are a legal expert fluent in Hindi and English.
The following is BNS Section {section_number} text (may contain Hindi and English):

{section_text[:1500]}

Please respond ONLY with a JSON object with these exact keys:
{{
  "english_summary": "2-sentence plain English summary of what this section says",
  "hindi_explanation": "2-sentence explanation in Hindi (Devanagari script)",
  "key_terms": ["term1 (हिंदी)", "term2 (हिंदी)"],
  "ipc_equivalent": "IPC section number this replaced, or 'New provision'"
}}"""

        for attempt in range(retries + 1):
            try:
                r = requests.post(
                    SARVAM_API_URL,
                    headers={
                        'Authorization': f'Bearer {SARVAM_API_KEY}',
                        'Content-Type':  'application/json',
                    },
                    json={
                        'model':    SARVAM_MODEL,
                        'messages': [{'role': 'user', 'content': prompt}],
                        'temperature': 0.1,
                        'max_tokens':  400,
                    },
                    timeout=30,
                )
                r.raise_for_status()
                content = r.json()['choices'][0]['message']['content']
                # Strip markdown code fences if present
                content = re.sub(r'^```json\s*|```$', '', content.strip(), flags=re.MULTILINE)
                parsed = json.loads(content)
                return parsed
            except (json.JSONDecodeError, KeyError, IndexError, TypeError) as _parse_err:
                raw_txt = ""
                try:
                    raw_txt = (r.json().get("choices") or [{}])[0].get("message", {}).get("content") or ""
                except Exception:
                    raw_txt = ""
                if not raw_txt and r is not None:
                    try:
                        raw_txt = r.text[:2000]
                    except Exception:
                        raw_txt = str(_parse_err)
                return {'english_summary': raw_txt[:4000], 'hindi_explanation': '',
                        'key_terms': [], 'ipc_equivalent': ''}
            except Exception as e:
                if attempt < retries:
                    time.sleep(1)
                else:
                    return {'english_summary': f'Error: {e}', 'hindi_explanation': '',
                            'key_terms': [], 'ipc_equivalent': ''}

    # Enrich a sample first to validate API key
    print(f'\n🔍 Testing Sarvam API with Section 1...')
    test = sarvam_enrich('1', raw_sections[0]['section_text'] if raw_sections else 'Test')
    if 'Error' in str(test.get('english_summary','')):
        print(f'  ⚠️  Sarvam API test failed: {test}')
        print('  Check SARVAM_API_KEY — get free key at https://dashboard.sarvam.ai')
        ENRICH = False
    else:
        print(f'  ✅ Sarvam API working')
        print(f'     Sample: {test.get("english_summary", "")[:100]}...')
        ENRICH = True

    # Enrich all sections (rate-limited: ~2 req/sec on free tier)
    enriched_sections = []
    ENRICH_LIMIT = len(raw_sections)   # set lower (e.g. 50) to save API credits

    for i, sec in enumerate(raw_sections[:ENRICH_LIMIT]):
        if ENRICH:
            enrichment = sarvam_enrich(sec['section_number'], sec['section_text'])
            time.sleep(0.5)   # polite rate limiting
        else:
            enrichment = {'english_summary': '', 'hindi_explanation': '',
                          'key_terms': [], 'ipc_equivalent': ''}

        enriched_sections.append({
            **sec,
            'english_summary':   enrichment.get('english_summary', ''),
            'hindi_explanation': enrichment.get('hindi_explanation', ''),
            'key_terms':         ', '.join(enrichment.get('key_terms', [])),
            'ipc_equivalent':    enrichment.get('ipc_equivalent', ''),
        })

        if (i+1) % 10 == 0:
            print(f'  Enriched {i+1}/{ENRICH_LIMIT} sections...', end='\r')

    print(f'\n  ✅ Enriched {len(enriched_sections)} sections with Sarvam-M')

    df_bns = pd.DataFrame(enriched_sections)


    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 4: Save to Delta
    # ═══════════════════════════════════════════════════════════════════════════

    if not df_bns.empty:
        print(f'\n📊 BNS sections summary:')
        print(f'   Total     : {len(df_bns)}  (BNS has 358 sections)')
        print(f'   Columns   : {list(df_bns.columns)}')
        print(f'   Enriched  : {"yes" if ENRICH else "no — set SARVAM_API_KEY"}')
        display(df_bns[['section_number','section_title',
                         'english_summary','hindi_explanation']].head(5))
        save_table(df_bns, 'bns_sections')

    print("""
ℹ️  For complete 358-section structured CSV (best for full RAG coverage):
   https://www.kaggle.com/datasets/nandr39/bharatiya-nyaya-sanhita-dataset-bns
   Download → Upload to Volume → load_uploaded_file('bns_sections.csv', 'bns_sections')
   Then re-run Sarvam enrichment on the uploaded CSV for full bilingual coverage.
""")

## 9c. BNS — Save sections to Delta

In [0]:
if not df_bns.empty:
    display(df_bns.head(5))
    save_table(df_bns, 'bns_sections')
else:
    print('⚠️  No BNS data available.')
    print('  Option A — Download CSV from Kaggle:')
    print('    kaggle.com/datasets/nandr39/bharatiya-nyaya-sanhita-dataset-bns')
    print('  Option B — Upload to Volume and run cell 12:')
    print('    load_uploaded_file("bns_sections.csv", "bns_sections")')

section_number section_title section_text source 2013 Explanation (a) persons falling under any of the descriptions made in this clause are public servants, whether appointed by the Government or not; (b) every person who is in actual possession of the situation of a public servant, whatever legal defect there may be in his right to hold that situation is a public servant; (c) “election” means an election for the purpose of selecting members of any legislative, municipal or other public authority, of whatever character, the method of selection to which is by, or under any law for the time being in force. Illustration. A Municipal Commissioner is a public servant; 10 of 1897. 18 of 2013. Sec. 1] THE GAZETTE OF INDIA EXTRAORDINARY 5 ____________________________________________________________ _____________________________________________________________ ____________________________________________________________ ___________________________________________________________ ________________________________________________________ _____________________________________________________________ (29) “reason to believe”.—A person is said to have “reason to believe” a thing, if he has sufficient cause to believe that thing but not otherwise; (30) “special law” means a law applicable to a particular subject; (31) “valuable security” means a document which is, or purports to be, a document whereby any legal right is created, extended, transferred, restricted, extinguished or released, or whereby any person acknowledges that he lies under legal liability, or has not a certain legal right. Illustration. A writes his name on the back of a bill of exchange. As the effect of this endorsement is to transfer the right to the bill to any person who may become the lawful holder of it, the endorsement is a “valuable security”; (32) “vessel” means anything made for the conveyance by water of human beings or of property; (33) “voluntarily”.—A person is said to cause an effect “voluntarily” when he causes it by means whereby he intended to cause it, or by means which, at the time of employing those means, he knew or had reason to believe to be likely to cause it. Illustration. A sets fire, by night, to an inhabited house in a large town, for the purpose of facilitating a robbery and thus causes the death of a person. Here, A may not have intended to cause death; and may even be sorry that death has been caused by his act; yet, if he knew that he was likely to cause death, he has caused death voluntarily; (34) “will” means any testamentary document; (35) “woman” means a female human being of any age; (36) “wrongful gain” means gain by unlawful means of property to which the person gaining is not legally entitled; (37) “wrongful loss” means the loss by unlawful means of property to which the person losing it is legally entitled; (38) “gaining wrongfully” and “losing wrongfully”.—A person is said to gain wrongfully when such person retains wrongfully, as well as when such person acquires wrongfully.Aperson is said to lose wrongfully when such person is wrongfully kept out of any property, as well as when such person is wrongfully deprived of property; and (39) words and expressions used but not defined in this Sanhita but defined in the InformationTechnologyAct, 2000 and the Bharatiya Nagarik Suraksha Sanhita,2023 shall have the meanings respectively assigned to them in that Act and Sanhita. 3. (1) Throughout this Sanhita every definition of an offence, every penal provision, and every Illustration of every such definition or penal provision, shall be understood subject to the exceptions contained in the Chapter entitled “General Exceptions”, though those exceptions are not repeated in such definition, penal provision, or Illustration. Illustrations. (a) The sections in this Sanhita, which contain definitions of offences, do not express that a child under seven years of age cannot commit such offences; but the definitions are to be understood subject to 

  💾 main.india_legal.bns_sections  (13 rows × 4 cols)


## 9d. BNS — IPC mapping table
Maps each BNS section to the IPC section it replaces.

In [0]:
print('📥 Fetching BNS ↔ IPC mapping...')
df_mapping = pd.DataFrame()
try:
    r = requests.get(
        'https://raw.githubusercontent.com/OpenNyAI/Opennyai/main/datasets/bns_ipc_mapping.csv',
        timeout=20, headers=HEADERS)
    r.raise_for_status()
    df_mapping = pd.read_csv(StringIO(r.text))
    print(f'  ✅ {len(df_mapping)} mapping rows from GitHub')
except Exception as e:
    print(f'  ⚠️  {e} — using curated stub')

if df_mapping.empty:
    df_mapping = pd.DataFrame([
        {'bns_section':'103','bns_title':'Murder',             'ipc_section':'302','ipc_title':'Punishment for murder',     'status':'Modified'},
        {'bns_section':'63', 'bns_title':'Rape',               'ipc_section':'375','ipc_title':'Rape',                      'status':'Modified'},
        {'bns_section':'111','bns_title':'Organised Crime',    'ipc_section':'None','ipc_title':'New section in BNS',       'status':'New'},
        {'bns_section':'113','bns_title':'Terrorist Act',      'ipc_section':'None','ipc_title':'New section in BNS',       'status':'New'},
        {'bns_section':'303','bns_title':'Theft',              'ipc_section':'378','ipc_title':'Theft',                    'status':'Retained'},
        {'bns_section':'318','bns_title':'Cheating',           'ipc_section':'415','ipc_title':'Cheating',                 'status':'Retained'},
        {'bns_section':'226','bns_title':'Attempt to Suicide', 'ipc_section':'309','ipc_title':'Attempt to commit suicide', 'status':'Modified'},
    ])
    print(f'  ✅ Using stub: {len(df_mapping)} rows')

save_table(df_mapping, 'bns_ipc_mapping')

📥 Fetching BNS ↔ IPC mapping...
  ⚠️  404 Client Error: Not Found for url: https://raw.githubusercontent.com/OpenNyAI/Opennyai/main/datasets/bns_ipc_mapping.csv — using curated stub
  ✅ Using stub: 7 rows
  💾 main.india_legal.bns_ipc_mapping  (7 rows × 5 cols)


## 10. Government Schemes — MyScheme (HuggingFace)
`shrijayan/gov_myscheme` — 700+ central/state schemes with eligibility & benefits.

In [0]:
print('📥 Fetching MyScheme dataset...')
df_schemes = pd.DataFrame()
HF_URLS = [
    'https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/data/train-00000-of-00001.parquet',
    'https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/schemes.parquet',
    'https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/schemes.csv',
]
for url in HF_URLS:
    try:
        r = requests.get(url, timeout=60, headers=HEADERS)
        r.raise_for_status()
        df_schemes = pd.read_parquet(BytesIO(r.content)) if url.endswith('.parquet') \
                     else pd.read_csv(StringIO(r.text))
        print(f'  ✅ {df_schemes.shape[0]} schemes from {url.split("/")[-1]}')
        break
    except Exception as e:
        print(f'  ⚠️  {url.split("/")[-1]}: {e}')

if df_schemes.empty:
    print('  Trying datasets library...')
    try:
        from datasets import load_dataset
        df_schemes = load_dataset('shrijayan/gov_myscheme', split='train').to_pandas()
        print(f'  ✅ {df_schemes.shape[0]} schemes via datasets library')
    except Exception as e:
        print(f'  ⚠️  {e}')

if not df_schemes.empty:
    display(df_schemes.head(5))
    save_table(df_schemes, 'gov_schemes_myscheme')
else:
    print('⚠️  Manual fallback:')
    print('   from datasets import load_dataset')
    print('   df = load_dataset("shrijayan/gov_myscheme")["train"].to_pandas()')

📥 Fetching MyScheme dataset...
  ⚠️  train-00000-of-00001.parquet: 404 Client Error: Not Found for url: https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/data/train-00000-of-00001.parquet
  ⚠️  schemes.parquet: 404 Client Error: Not Found for url: https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/schemes.parquet
  ⚠️  schemes.csv: 404 Client Error: Not Found for url: https://huggingface.co/datasets/shrijayan/gov_myscheme/resolve/main/schemes.csv
  Trying datasets library...
  ⚠️  No module named 'datasets'
⚠️  Manual fallback:
   from datasets import load_dataset
   df = load_dataset("shrijayan/gov_myscheme")["train"].to_pandas()


## 11. Combine into RAG corpus
Merges BNS sections + IPC mapping + gov schemes into one `legal_rag_corpus` table
with `chunk_id`, `source`, `doc_type`, `title`, `text` — ready for embedding.

In [0]:
corpus = []

# BNS sections (prefer english_summary for RAG when present)
try:
    df_b = spark.table(f'{CATALOG}.{SCHEMA}.bns_sections').toPandas()
    tc  = next((c for c in df_b.columns if 'text' in c.lower() or 'content' in c.lower()), df_b.columns[-1])
    nc  = next((c for c in df_b.columns if 'number' in c.lower() or c.lower() == 'section'), None)
    ttc = next((c for c in df_b.columns if 'title' in c.lower()), None)
    has_summary = 'english_summary' in df_b.columns
    for _, row in df_b.iterrows():
        num = str(row[nc]) if nc else ''
        if has_summary and pd.notna(row.get('english_summary')) and str(row.get('english_summary', '')).strip():
            body = str(row['english_summary'])[:2000]
        else:
            body = str(row[tc])[:2000]
        corpus.append({'chunk_id': f'BNS_S{num}', 'source': 'BNS_2023', 'doc_type': 'criminal_law',
                       'title': f'BNS Section {num}: {str(row[ttc]) if ttc else ""}',
                       'text': body})
    print(f'  ✅ BNS: {len(df_b)} sections')
except Exception as e:
    print(f'  ⚠️  BNS sections: {e}')

# BNS-IPC mapping
try:
    df_m = spark.table(f'{CATALOG}.{SCHEMA}.bns_ipc_mapping').toPandas()
    for _, row in df_m.iterrows():
        corpus.append({'chunk_id': f'MAP_BNS{row.get("bns_section","?")}', 'source': 'BNS_IPC_MAPPING',
                       'doc_type': 'law_mapping',
                       'title': f'BNS {row.get("bns_section","")} replaces IPC {row.get("ipc_section","")}',
                       'text': (f'BNS Section {row.get("bns_section","")} ({row.get("bns_title","")}) '
                                f'corresponds to IPC {row.get("ipc_section","")} ({row.get("ipc_title","")}). '
                                f'Status: {row.get("status","")}')})
    print(f'  ✅ Mapping: {len(df_m)} rows')
except Exception as e:
    print(f'  ⚠️  Mapping: {e}')

# Gov Schemes
try:
    df_s  = spark.table(f'{CATALOG}.{SCHEMA}.gov_schemes_myscheme').toPandas()
    n_col = next((c for c in df_s.columns if 'name' in c.lower() or 'scheme' in c.lower()), df_s.columns[0])
    d_col = next((c for c in df_s.columns if 'desc' in c.lower()), None)
    e_col = next((c for c in df_s.columns if 'elig' in c.lower()), None)
    b_col = next((c for c in df_s.columns if 'bene' in c.lower()), None)
    for i, row in df_s.iterrows():
        parts = [f'Scheme: {row[n_col]}']
        if d_col and pd.notna(row.get(d_col)): parts.append(f'Description: {row[d_col]}')
        if e_col and pd.notna(row.get(e_col)): parts.append(f'Eligibility: {row[e_col]}')
        if b_col and pd.notna(row.get(b_col)): parts.append(f'Benefits: {row[b_col]}')
        corpus.append({'chunk_id': f'SCHEME_{i:04d}', 'source': 'MYSCHEME_GOV',
                       'doc_type': 'government_scheme', 'title': str(row[n_col]),
                       'text': ' | '.join(parts)[:2000]})
    print(f'  ✅ Schemes: {len(df_s)} rows')
except Exception as e:
    print(f'  ⚠️  Schemes: {e}')

if corpus:
    df_corpus = pd.DataFrame(corpus)
    save_table(df_corpus, 'legal_rag_corpus')
    print(f'\n🏛️  legal_rag_corpus: {len(df_corpus)} total chunks')
    display(df_corpus.groupby('source')['chunk_id'].count().reset_index(name='chunks'))
else:
    print('⚠️  No chunks — run sections 9 and 10 first.')

## 12. Manual upload helper
Upload files via **Catalog → Volumes → legal_files → Upload to this volume**,
then call `load_uploaded_file(filename, table_name)` below.

In [0]:
def load_uploaded_file(filename, table_name, fmt='csv', skip_rows=0):
    """
    Load a file from Unity Catalog Volume into a Delta table.
    Upload path: Catalog → main → india_legal → Volumes → legal_files → Upload

    Args:
        filename   : just the filename, e.g. 'daksh_hc.csv'
        table_name : Delta table name (no schema prefix)
        fmt        : 'csv' | 'excel' | 'parquet'
        skip_rows  : header rows to skip (NCRB Excel needs skip_rows=3)
    """
    path = f'{VOL_PATH}/{filename}'
    print(f'📂 Loading {path} → {CATALOG}.{SCHEMA}.{table_name}')
    try:
        if fmt == 'csv':
            pdf = spark.read.option('header','true').option('inferSchema','false') \
                       .option('multiLine','true').csv(path).toPandas()
        elif fmt == 'excel':
            pdf = pd.read_excel(path, engine='openpyxl', skiprows=skip_rows)
            pdf.dropna(how='all', inplace=True)
            pdf.dropna(axis=1, how='all', inplace=True)
        elif fmt == 'parquet':
            pdf = spark.read.parquet(path).toPandas()
        else:
            raise ValueError(f'Unsupported format: {fmt}')
        save_table(pdf, table_name)
    except Exception as e:
        print(f'  ⚠️  {e}')

# Examples — uncomment after uploading:
# load_uploaded_file('daksh_hc_cases.csv',  'daksh_hc_cases')
# load_uploaded_file('njdg_state.csv',      'njdg_state_pendency')
# load_uploaded_file('ncrb_ipc_2022.xlsx',  'ncrb_ipc_2022', fmt='excel', skip_rows=3)
# load_uploaded_file('bns_sections.csv',    'bns_sections')
print('✅ load_uploaded_file() ready')

## 13. Sample queries
Run each cell independently to verify your tables.

In [0]:
print(f'📋 All tables in {CATALOG}.{SCHEMA}:')
display(spark.sql(f'SHOW TABLES IN {CATALOG}.{SCHEMA}'))

In [0]:
display(spark.sql("""
    SELECT Name, Party, State, Questions, Attendance
    FROM   main.india_legal.mp_activity_loksabha18
    ORDER  BY CAST(Questions AS DOUBLE) DESC
    LIMIT  10
"""))

In [0]:
display(spark.sql("""
    SELECT status, COUNT(*) AS count
    FROM   main.india_legal.prs_bills
    GROUP  BY status
    ORDER  BY count DESC
"""))

In [0]:
display(spark.sql("""
    SELECT year, COUNT(*) AS acts_passed
    FROM   main.india_legal.prs_acts
    WHERE  year IS NOT NULL AND year != ''
    GROUP  BY year
    ORDER  BY year DESC
    LIMIT  20
"""))

In [0]:
display(spark.sql("""
    SELECT chunk_id, title, LEFT(text, 300) AS preview
    FROM   main.india_legal.legal_rag_corpus
    WHERE  source = 'BNS_2023'
      AND  LOWER(title) LIKE '%organised%'
"""))

In [0]:
display(spark.sql("""
    SELECT title, LEFT(text, 400) AS preview
    FROM   main.india_legal.legal_rag_corpus
    WHERE  source = 'MYSCHEME_GOV'
      AND  LOWER(text) LIKE '%women%'
    LIMIT  10
"""))

In [0]:
display(spark.sql("""
    SELECT title, total_records, org, keyword
    FROM   main.india_legal.datagov_catalog
    ORDER  BY CAST(total_records AS INT) DESC
    LIMIT  20
"""))